# Exercise: Introduction to graphs and graph signals
In this notebook you'll explore the NetworkX and Pytorch Geometric libraries, used to work with graphs.

# Import libraries

In [ ]:
import torch
import numpy as np
import networkx as nx
torch.manual_seed(0)

import torch_geometric as pyg
from torch_geometric.data import Data
from torch_geometric.utils.convert import to_networkx
from torch_geometric.datasets import TUDataset, Planetoid
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt

print('PyTorch Geometric version:',pyg.__version__)

# What is a graph?

A **graph** $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ is a collection of nodes $\mathcal{V} \in \mathbb{R}^N$ and edges $\mathcal{E} \in \mathbb{R}^E$.

We can define the connectivity of the graph via an **adjacency matrix** $\mathbf{A} \in \mathbb{R}^{N \times N}$, in which each entry $A_{ij}$ is either $1$ if edge $ij$ exists or $0$ otherwise.
For example, $\mathbf{A}$ can look like:

$$
\mathbf{A} = 
\begin{bmatrix}
0 & 1 & 1 & 0 \\
1 & 0 & 1 & 1 \\
1 & 1 & 0 & 1 \\
0 & 1 & 1 & 0 \\
\end{bmatrix}
$$

For computational purposes, we can also use a sparse representation of the connectivity, using an edge list or **edge index** $\mathbf{E} \in \mathbb{R}^{E \times 2}$.
This matrix defines which node are connected to which other. For each row, the first value is referred as 'source node' (src) and the second one as 'destination node' (dst).

Let's begin by creating a very simple directed graph. 
We do so by defining how each node is connected to each other.

In [ ]:
# define which nodes are connected [src, dst]
edge_index = torch.tensor([[0, 1],  # 0->1
                           [1, 3],  # 1->3
                           [0, 2],  # 0->2
                           [2, 0],  # 2->0
                           [2, 3]], # 2->3
                          dtype=torch.long)

print(edge_index)
print(f"We have {len(edge_index)} edges so our edge index is of size: {edge_index.shape}")

In [ ]:
# define position of each node (this is useful only for graphical purposes)
# notation is {node_id : (x,y)}
pos = {0: (0,0),
       1: (1,0),
       2: (0,1),
       3: (1,1)}

In [ ]:
# create graph using networkx
graph = nx.DiGraph() #directed graph
graph.add_edges_from(edge_index.tolist())

fig, ax = plt.subplots(1,1,figsize=(3,3))

nx.draw(graph, pos, with_labels=True, node_size=500, font_size=15, ax=ax)

## Graph shift operators

Thanks to the networkx library, we can easily determine several graph shift operators $\mathbf{S}$, as we saw in the lectures.
A graph shift operator (GSO) is any squared matrix in which $S_{ij} \neq 0$ only if an edge exists between nodes $i$ and $j$ or if $i=j$.

Among this class of matrices we can find the adjacency matrix and the Laplacian matrix.

In [ ]:
# Adjacency matrix
adj_matrix = nx.adjacency_matrix(graph).todense()

print("Adjacency matrix:")
print(adj_matrix)

The Laplacian matrix can be obtained as a subtraction of the adjacency matrix from the degree matrix. However, for directed graph the degree matrix is split in two components: in-degree, representing the degree of incoming edges, and out-degree, representing the degree of outgoing edges.

To simplify things, we will first convert the directed graph we created in an undirected graph.

We can then determine the Laplacian as:

$$
\mathbf{L} = \mathbf{D} - \mathbf{A}
$$

where $\mathbf{A}$ is the adjacency matrix and $\mathbf{D}$ is the degree matrix.

In [ ]:
# Convert directed graph to undirected graph
graph = graph.to_undirected()
adj_matrix = nx.adjacency_matrix(graph).todense()

# Degree matrix
degree_matrix = np.diag(np.sum(adj_matrix, axis=1))

# Laplacian matrix
laplacian_matrix = degree_matrix - adj_matrix

print("Laplacian matrix:")
print(laplacian_matrix)

In [ ]:
# Or use networkx functions
laplacian_matrix = nx.laplacian_matrix(graph).todense()

print("Laplacian matrix:")
print(laplacian_matrix)


**Exercise:**

Create a function to evaluate the normalized adjacency matrix, which is defined as:

$$
\mathbf{A}' = \mathbf{\tilde{D}}^{-\frac{1}{2}} \mathbf{\tilde{A}} \mathbf{\tilde{D}}^{-\frac{1}{2}}
$$

where $\mathbf{\tilde{A}} = \mathbf{A} + \mathbf{I}$ and $\tilde{D}_{ii} = \sum_j{\tilde{A}_{ij}}$.


In [ ]:
def normalized_adjacency_matrix(adj_matrix):
    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #
    return norm_adj_matrix

norm_adj_matrix = normalized_adjacency_matrix(adj_matrix)

print("Normalized adjacency matrix:")
print(norm_adj_matrix)

# Graph signals

A graph signal $\mathbf{x} \in \mathbb{R}^N$ represents a value defined on the nodes of a graph. The $i^{th}$ entry $x_i$ represents the value associated to node $i$.

In case of multiple features per node, we can also define a graph signal matrix (or node feature matrix) $\mathbf{X} \in \mathbb{R}^{N \times F}$, where $F$ is the number of features.

In [ ]:
# define unitary signals on the nodes
x = torch.tensor([[1], [2], [-0.5], [6]], dtype=torch.float32)

print(f"The node feature matrix has size: {x.shape}")
print(x)

## Pytorch geometric

We can easily associate this matrix to a graph using the PyTorch Geometric (pyg) library.

Pyg provides an object called Data, which acts similarly to a dictionary.
This is used to store the node feature matrix, the edge list, and any other feature that might be useful to describe the graph, such as edge features (generally called adge attributes in the Data object).

The Data object also has some useful attributes and methods.

In [ ]:
# Data object is similar to a dictionary
data = Data(x=x, edge_index=edge_index.t().contiguous())
print(data)

In [ ]:
print(f"Number of nodes in the graph: {data.num_nodes}\n")

print(f"Number of edges in the graph: {data.num_edges}\n")

print(f"Number of features per node in the node feature matrix: {data.num_node_features}\n")

print(f"Does the graph have isolated nodes (nodes with no edges)? {data.has_isolated_nodes()}\n")

print(f"Does the graph have self-loops (edges that connect a node to itself)? {data.has_self_loops()}\n")

print(f"Is the graph directed? {data.is_directed()}\n")

# Loading graph datasets

There are plenty of datasets provided by pyg that are ready to use.

In the following cells, we will consider the TUDataset which consists of several graph benchmark datasets, collected from TU Dortmund University. 
In particular, we will use the ENZYMES dataset, which contains 600 enzymes (molecules) represented as graphs.
This dataset is used for graph classification.

For more datasets and their corresponding learning task you can check https://pytorch-geometric.readthedocs.io/en/latest/cheatsheet/data_cheatsheet.html

In [ ]:
dataset = TUDataset(root='tmp/ENZYMES', name='ENZYMES', use_node_attr=True)
print(f"This dataset has {len(dataset)} graphs")

# number of classes
print(f"Number of classes: {dataset.num_classes}")

# number of features per node
print(f"Number of features per node: {dataset.num_node_features}")

In [ ]:
# Let's explore one sample of graph
data = dataset[0]

# we can convert the graph to a networkx graph
graph = to_networkx(data, to_undirected=True)

# since we don't know the position of the nodes, we use a spring layout
pos = nx.spring_layout(graph, seed=42)

fig, ax = plt.subplots(1,1,figsize=(6,6))
nx.draw(graph, pos, with_labels=True, node_size=100, font_size=10, ax=ax)
plt.show()


# Graph analysis

Given a graph, we can determine some properties that might be useful in understanding the importance of some nodes or structures in the graph.

**Degree Centrality:** counts the number of edges that a node has. Nodes with a high degree centrality have many connections.

In [ ]:
# plot a graph where the size of each node is proportional to its degree
fig, ax = plt.subplots(1,1,figsize=(6,6))

node_size = [v**3 for v in dict(graph.degree()).values()]

plt.title("Node size proportional to its degree")
nx.draw(graph, pos, with_labels=True, node_size=node_size, font_size=10, ax=ax)
plt.show()

Consider the Cora dataset. This is a citation network (nodes represent papers and edges connect cited papers).
Differently from the previous dataset, this consists of only one big graph. 
This dataset is used for node classification.
Each node is described by a 1433-dimensional bag-of-words feature vector. Two documents are connected if there exists a citation link between them. The task is to infer the category of each document (7 in total).

**Exercise:**

1. Print the following statistics about the graph: number of graphs, number of features, number of classes, number of nodes, number of edges, average node degree.

There is also a new attribute specific to this type of task: train_mask. What do you think does this represent?

2. Plot the graph, indicating with colors which nodes are used for training, validation, and testing

In [ ]:
dataset = Planetoid(root='tmp/Planetoid', name='Cora')

# Dataset statistics
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

In [ ]:
# Plot the graph with separate colors for training, validation, and test nodes
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

You might notice that not all nodes are plotted in this way. Why?

<!-- ---------------------- student exercise --------------------------------- -->
Answer: we have no information about some nodes in the graph, as highlighted by the mask arrays
<!-- ---------------------- student exercise --------------------------------- -->


In [ ]:
# Why are there missing nodes in the plot? Answer by telling how many nodes are missing
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

# Graph mini-batching

PyG achieves parallelization over a mini-batch by creating sparse block diagonal adjacency matrices (defined by edge_index) and concatenating feature and target matrices in the node dimension. This composition allows differing number of nodes and edges over examples in one batch:

$$
\begin{split}\mathbf{A} = \begin{bmatrix} \mathbf{A}_1 & & \\ & \ddots & \\ & & \mathbf{A}_n \end{bmatrix}, \qquad \mathbf{X} = \begin{bmatrix} \mathbf{X}_1 \\ \vdots \\ \mathbf{X}_n \end{bmatrix}, \qquad \mathbf{Y} = \begin{bmatrix} \mathbf{Y}_1 \\ \vdots \\ \mathbf{Y}_n \end{bmatrix}\end{split}
$$

The obtained Batch class has a few important attributes which allow to distinguish between the batched graphs.

In [ ]:
loader = DataLoader(dataset, batch_size=4, shuffle=False)

batch = next(iter(loader))
print(batch)

In [ ]:
print(f"Number of graphs in the batch: {batch.num_graphs}\n")

print("Index of the graph in the batch:")
print(batch.ptr,"\n")

print(f"Which graph does each node belong to?")
print(batch.batch)

Batch.batch is a column vector which maps each node to its respective graph in the batch:
$$
\mathrm{batch} = {\begin{bmatrix} 0 & \cdots & 0 & 1 & \cdots & n - 2 & n -1 & \cdots & n - 1 \end{bmatrix}}^{\top}
$$

This resource will be more relevant when we start training our graph neural network models, in the following notebook.

# Additional resources and references

https://pytorch-geometric.readthedocs.io/en/latest/